## Introduction

In this lab assessment, you'll practice your knowledge of JOIN statements and subqueries, using various types of joins and various methods for specifying the links between them. One of the main benefits of using a relational database is the table relations that define them which allow you to access and connect data together via shared columns. By writing more advanced SQL queries that utilize joins and subqueries you can provide a deeper and more granular level of analysis and data retrieval.

This assessment will continue looking at the familiar Northwind database that contains customer relationship management (CRM) data as well as employee and product data. You will take a deeper dive into this database in order to accomplish more advanced SQL queries that require you to access data from multiple tables at once. 

Imagine that you are working in an analyst role for the sales rep team. They have collaborated with the customer relations and the product teams to take a comprehensive look at the employee to customer pipeline in an attempt to find areas of improvement and potential growth. You have been asked to provide some specific data and statistics regarding this project.

## Learning Objectives

You will be able to:

* Write SQL queries that make use of various types of joins
* Choose and perform whichever type of join is best for retrieving desired data
* Write subqueries to decompose complex queries

## Database

The database will be the customer relationship management (CRM) database, which has the following ERD.

![Database-Schema.png](ERD.png)

### Connect to the database

In the cell below we have provided the code to import both pandas and sqlite3 as well as define and create the connection to the database you will use. Also displayed is the schema and table names from the database. Use this information in conjunction with the ERD image above to assist in creating your SQL Queries.

Major Hint: Look for the shared columns across tables you need to 'join' together.

In [83]:
# CodeGrade step0
# Run this cell without changes

# SQL Library and Pandas Library
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect('data.sqlite')

pd.read_sql("""SELECT * FROM sqlite_master""", conn)
#pd.read_sql("""SELECT * FROM products""", conn)
#pd.read_sql("""SELECT * FROM orderdetails""", conn)
pd.read_sql("""SELECT * FROM offices""", conn)

,officeCode,city,phone,addressLine1,addressLine2,state,country,postalCode,territory
0,1,San Francisco,+1 650 219 4782,100 Market Street,Suite 300,CA,USA,94080,NA
1,2,Boston,+1 215 837 0825,1550 Court Place,Suite 102,MA,USA,02107,NA
2,3,NYC,+1 212 555 3000,523 East 53rd Street,apt. 5A,NY,USA,10022,NA
3,4,Paris,+33 14 723 4404,43 Rue Jouffroy D'abbans,,,France,75017,EMEA
4,5,Tokyo,+81 33 224 5000,4-1 Kioicho,,Chiyoda-Ku,Japan,102-8578,Japan
5,6,Sydney,+61 2 9264 2451,5-11 Wentworth Avenue,Floor #2,,Australia,NSW 2010,APAC
6,7,London,+44 20 7877 2041,25 Old Broad Street,Level 7,,UK,EC2N 1HN,EMEA


## Part 1: Join and Filter

### Step 1

The company would like to let Boston employees go remote but need to know more information about who is working in that office. Return the first and last names and the job titles for all employees in Boston.

In [84]:
# CodeGrade step1
# Replace None with your code
df_boston = pd.read_sql("""
    SELECT firstName, lastName, jobTitle
    FROM employees e
    JOIN offices o ON e.officeCode = o.officeCode  
    WHERE o.city= "Boston"               
""",conn)

### Step 2

Recent downsizing and employee attrition have caused some mixups in office tracking and the company is worried they are supporting a 'ghost' location. Are there any offices that have zero employees?

In [85]:
# CodeGrade step2
# Replace None with your code
df_zero_emp = pd.read_sql("""
    SELECT COUNT(DISTINCT e.employeeNumber) AS emp_num, o.officeCode, o.city
    FROM offices o
    JOIN employees e ON o.officeCode = e.officeCode  
    GROUP BY o.officeCode
    HAVING emp_num = 0                                  
""",conn)

df_zero_emp

,emp_num,officeCode,city


## Part 2: Type of Join

### Step 3

As a part of this larger analysis project the HR department is taking the time to audit employee records to make sure nothing is out of place and have asked you to produce a report of all employees. Return the employees first name and last name along with the city and state of the office that they work out of (if they have one). Include all employees and order them by their first name, then their last name.

In [86]:
# CodeGrade step3
# Replace None with your code
df_employee = pd.read_sql("""
    SELECT e.firstName, e.lastName, o.city, o.state
    FROM employees e
    JOIN offices o ON e.officeCode = o.officeCode
    ORDER BY e.firstName, e.lastName                                      
""",conn)

### Step 4
The customer management and sales rep team know that they have several 'customers' in the system that have not placed any orders. They want to reach out to these customers with updated product catalogs to try and get them to place initial orders. Return all of the customer's contact information (first name, last name, and phone number) as well as their sales rep's employee number for any customer that has not placed an order. Sort the results alphabetically based on the contact's last name

There are several approaches you could take here, including a left join and filtering on null values or using a subquery to filter out customers who do have orders. In total there are 24 customers who have not placed an order.

In [87]:
# CodeGrade step4
# Replace None with your code
df_contacts = pd.read_sql("""
    SELECT c.contactFirstName, c.contactLastName,  c.phone, c.salesRepEmployeeNumber
    FROM customers c
    LEFT JOIN orders o ON c.customerNumber=o.customerNumber
    GROUP BY c.customerNumber
    HAVING COUNT(orderNumber) = 0
    ORDER BY c.contactLastName 
""",conn)

## Part 3: Built-in Function

### Step 5

The accounting team is auditing their figures and wants to make sure all customer payments are in alignment, they have asked you to produce a report of all the customer contacts (first and last names) along with details for each of the customers' payment amounts and date of payment. They have asked that these results be sorted in descending order by the payment amount.

Hint: A member of their team mentioned that they are not sure the 'amount' column is being stored as the right datatype so keep this in mind when sorting.

In [88]:
# CodeGrade step5
# Replace None with your code
df_payment = pd.read_sql("""
    SELECT c.contactFirstName, c.contactLastName, CAST(p.amount AS REAL) , p.paymentDate
    FROM payments p
    LEFT JOIN customers c ON p.customerNumber = c.customerNumber  
    ORDER BY CAST(p.amount AS REAL) DESC                 
""",conn)

## Part 4: Joining and Grouping

### Step 6

The sales rep team has noticed several key team members that stand out as having trustworthy business relations with their customers, reflected by high credit limits indicating more potential for orders. The team wants you to identify these 4 individuals. Return the employee number, first name, last name, and number of customers for employees whose customers have an average credit limit over 90k. Sort by number of customers from high to low.

In [89]:
# CodeGrade step6
# Replace None with your code
df_credit = pd.read_sql("""
    SELECT e.employeeNumber, e.firstName, e.lastName, COUNT(c.customerNumber)
    FROM employees e
    LEFT JOIN customers c ON e.employeeNumber= c.salesRepEmployeeNumber
    GROUP BY e.employeeNumber
    HAVING AVG(c.creditLimit) > 90000
    ORDER BY COUNT(c.customerNumber) DESC                                                            
""", conn)

df_credit

,employeeNumber,firstName,lastName,COUNT(c.customerNumber)
0,1501,Larry,Bott,8
1,1370,Gerard,Hernandez,7
2,1165,Leslie,Jennings,6
3,1612,Peter,Marsh,5


### Step 7

The product team is looking to create new model kits and wants to know which current products are selling the most in order to get an idea of what is popular. Return the product name and count the number of orders for each product as a column named 'numorders'. Also return a new column, 'totalunits', that sums up the total quantity of product sold (use the quantityOrdered column). Sort the results by the totalunits column, highest to lowest, to showcase the top selling products.

In [90]:
# CodeGrade step7
# Replace None with your code
df_product_sold = pd.read_sql("""
    SELECT p.productName, COUNT(od.quantityOrdered) AS numorders,SUM(quantityOrdered) AS totalunits
    FROM products p
    LEFT JOIN orderdetails od ON p.productCode = od.productCode 
    GROUP BY p.productName
    ORDER BY totalunits DESC                                                                                        
""", conn)

df_product_sold

,productName,numorders,totalunits
0,1992 Ferrari 360 Spider red,53,1808.0
1,1937 Lincoln Berline,28,1111.0
2,American Airlines: MD-11S,28,1085.0
3,1941 Chevrolet Special Deluxe Cabriolet,28,1076.0
4,1930 Buick Marquette Phaeton,28,1074.0
...,...,...,...
105,1911 Ford Town Car,25,832.0
106,1936 Mercedes Benz 500k Roadster,25,824.0
107,1970 Chevy Chevelle SS 454,25,803.0
108,1957 Ford Thunderbird,24,767.0


## Part 5: Multiple Joins

### Step 8

As a follow-up to the above question, the product team also wants to know how many different customers ordered each product to get an idea of market reach. Return the product name, code, and the total number of customers who have ordered each product, aliased as 'numpurchasers'. Sort the results by the highest  number of purchasers.

Hint: You might need to join more than 2 tables. Use DISTINCT to return unique/different values.

In [91]:
# CodeGrade step8
# Replace None with your code
df_total_customers = pd.read_sql("""
    SELECT p.productName,p.productCode, COUNT(DISTINCT o.customerNumber) AS numpurchasers
    FROM products p
    LEFT JOIN orderdetails od ON p.productCode = od.productCode
    LEFT JOIN orders o ON od.orderNumber = o.orderNumber
    LEFT JOIN customers c ON o.customerNumber = c.customerNumber                                                                                          
    GROUP BY p.productName
    ORDER BY numpurchasers DESC                                                                    
""",conn)

df_total_customers

,productName,productCode,numpurchasers
0,1992 Ferrari 360 Spider red,S18_3232,40
1,Boeing X-32A JSF,S72_1253,27
2,1972 Alfa Romeo GTA,S10_4757,27
3,1952 Alpine Renault 1300,S10_1949,27
4,1934 Ford V8 Coupe,S18_2957,27
...,...,...,...
105,2002 Chevy Corvette,S24_3432,18
106,1969 Chevrolet Camaro Z28,S24_3191,18
107,1952 Citroen-15CV,S24_2887,18
108,1949 Jaguar XK 120,S24_2766,18


### Step 9

The custom relations team is worried they are not staffing locations properly to account for customer volume. They want to know how many customers there are per office. Return the count as a column named 'n_customers'. Also return the office code and city.

In [92]:
# CodeGrade step9
# Replace None with your code
df_customers = pd.read_sql("""
    SELECT COUNT(customerNumber) AS n_customers, of.officeCode, of.city
    FROM offices of
    LEFT JOIN employees e ON of.officeCode = e.officeCode
    LEFT JOIN customers c ON e.employeeNumber = c.salesRepEmployeeNumber
    GROUP BY of.officeCode                       
""",conn)

df_customers

,n_customers,officeCode,city
0,12,1,San Francisco
1,12,2,Boston
2,15,3,NYC
3,29,4,Paris
4,5,5,Tokyo
5,10,6,Sydney
6,17,7,London


## Part 6: Subquery

### Step 10

Having looked at the results from above, the product team is curious to dig into the underperforming products. They want to ask members of the team who have sold these products about what kind of messaging was successful in getting a customer to buy these specific products. Using a subquery or common table expression (CTE), select the employee number, first name, last name, city of the office, and the office code for employees who sold products that have been ordered by fewer than 20 customers.

Hint: Start with the subquery, find all the products that have been ordered by 19 or less customers, consider adapting one of your previous queries.

In [93]:
# CodeGrade step10
# Replace None with your code
df_under_20 = pd.read_sql("""
    SELECT e.employeeNumber, e.firstName, e.lastName, of.city, of.officeCode
    FROM employees e
    LEFT JOIN offices of ON e.officeCode = of.officeCode
    WHERE e.employeeNumber IN(                                                 
         SELECT c.salesRepEmployeeNumber
         FROM products p
         LEFT JOIN orderdetails od ON p.productCode = od.productCode
         LEFT JOIN orders o ON od.orderNumber = o.orderNumber
         LEFT JOIN customers c ON o.customerNumber = c.customerNumber                                                                                          
         GROUP BY p.productName, c.salesRepEmployeeNumber
         HAVING COUNT(DISTINCT o.customerNumber) <= 19
    )
    ORDER BY e.employeeNumber                                                   

""",conn)
df_under_20

,employeeNumber,firstName,lastName,city,officeCode
0,1165,Leslie,Jennings,San Francisco,1
1,1166,Leslie,Thompson,San Francisco,1
2,1188,Julie,Firrelli,Boston,2
3,1216,Steve,Patterson,Boston,2
4,1286,Foon Yue,Tseng,NYC,3
5,1323,George,Vanauf,NYC,3
6,1337,Loui,Bondur,Paris,4
7,1370,Gerard,Hernandez,Paris,4
8,1401,Pamela,Castillo,Paris,4
9,1501,Larry,Bott,London,7


### Close the connection

In [94]:
# Run this cell without changes

conn.close()